# Student Dropout Prediction - Preprocessing

This notebook prepares the final MVP dataset. It removes `Enrolled`, encodes the target, keeps the fixed 10 enrollment/background features, and saves `processed.csv`.

In [ ]:
# Import libraries used for preprocessing.
from pathlib import Path
import json

import pandas as pd

## Load Data

Resolve project paths and load the raw UCI student dropout dataset.

In [ ]:
# Resolve paths so the notebook works from either the project root or the notebooks folder.
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Feature config path:", FEATURE_CONFIG_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)

In [ ]:
# Load raw data and feature config.
df = pd.read_csv(DATA_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

print("Raw dataset shape:", df.shape)
print("Raw target distribution:")
print(df["Target"].value_counts())

df.head()

## Final MVP Feature Scope

The same 10 features are used from preprocessing through model training and the Streamlit app. Semester variables, macroeconomic fields, application mode/order, and post-acceptance/admin variables are excluded to reduce leakage and keep the MVP form understandable.

In [ ]:
# Define final candidate MVP features from app/feature_config.json.
candidate_mvp_features = feature_config["features"]

print("Final MVP feature count:", len(candidate_mvp_features))
print(candidate_mvp_features)

## Filter Rows and Encode Target

`Enrolled` is removed because the project is a binary classification task: `Graduate` vs `Dropout`. Dropout is encoded as the positive class.

In [ ]:
# Remove Enrolled and encode the target.
df_processed = df[df["Target"] != "Enrolled"].copy()

target_mapping = {
    "Graduate": 0,
    "Dropout": 1
}

df_processed["Target"] = df_processed["Target"].map(target_mapping)

print("Dataset shape after removing Enrolled:", df_processed.shape)
print("Encoded target distribution:")
print(df_processed["Target"].value_counts().sort_index())

## Keep Final Columns

The saved processed dataset contains only the 10 fixed MVP features plus the encoded target.

In [ ]:
# Keep only selected features and target.
missing_features = [
    feature for feature in candidate_mvp_features
    if feature not in df_processed.columns
]

if missing_features:
    raise ValueError(f"Missing MVP features from dataset: {missing_features}")

df_processed = df_processed[candidate_mvp_features + ["Target"]].copy()

print("Processed dataset shape:", df_processed.shape)
df_processed.head()

## Save Processed Dataset

This file is the source for model training and app input options.

In [ ]:
# Save final processed dataset.
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

print("Processed dataset saved to:", PROCESSED_DATA_PATH)
print("Saved shape:", df_processed.shape)

## Preprocessing Summary

- `Enrolled` rows are removed.
- Target is encoded as `Graduate = 0`, `Dropout = 1`.
- Final output has 10 MVP features plus `Target`.
- No semester, macroeconomic, application mode/order, or post-acceptance/admin variables are kept.